# Three-mass / four-spring chain — compose-physics validation

End-to-end correctness witness for the rewrite. Registers the problem via the Python CLI, loads the resulting `.so` with `ctypes`, and verifies:

1. **Eigenmode FFT match** — three analytical normal modes recovered from `compute_update` time-stepping.
2. **Residual at machine zero** — `compute_residual` returns ~0 on every step's output.
3. **Lisp-vs-C bit identity** — funcall-evaluated trees match compiled C output exactly.
4. **Energy bounded** — symplectic Euler conserves total energy within its expected oscillation band.

The notebook is purely a validator. It never edits the project; it only invokes the CLI and reads the artifacts it produced.

In [1]:
import os, sys, ctypes, subprocess, shutil, math
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'compose-physics.asd').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('cannot locate project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
REGISTRY_ROOT = Path('/tmp/compose-physics-validation')
PROBLEM_LISP = PROJECT_ROOT / 'notebooks/three-mass-four-spring.lisp'
PROJECT_ROOT, PROBLEM_LISP, REGISTRY_ROOT

(PosixPath('/home/javier/Desktop/Computational_Physics/compose-physics'),
 PosixPath('/home/javier/Desktop/Computational_Physics/compose-physics/notebooks/three-mass-four-spring.lisp'),
 PosixPath('/tmp/compose-physics-validation'))

## Register the problem via the CLI

In [2]:
if REGISTRY_ROOT.exists():
    shutil.rmtree(REGISTRY_ROOT)
REGISTRY_ROOT.mkdir(parents=True)
completed = subprocess.run(
    [sys.executable, '-m', 'compose_physics', 'register',
     str(PROBLEM_LISP),
     '--registry-root', str(REGISTRY_ROOT),
     '--source-dest', 'copy',
     '--c-flags', '-O2 -fPIC -fno-fast-math -fno-associative-math'],
    cwd=PROJECT_ROOT, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode != 0:
    print('STDERR:', completed.stderr)
    raise RuntimeError('registration failed')

> registering three-mass-four-spring.lisp into /tmp/compose-physics-validation
> building shared library with 16 job(s)
╭─────────────────────────── registration complete ────────────────────────────╮
│ Problem name   three-mass-four-spring                                        │
│ Content hash   c8fb38ba6ceaff018094b31c904c1ba0                              │
│ Directory      /tmp/compose-physics-validation/c8fb38ba6ceaff018094b31c904c… │
│ Library        libthree-mass-four-spring.so                                  │
│ Chunk sources  2                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯



In [3]:
problem_dirs = [p for p in REGISTRY_ROOT.iterdir() if p.is_dir()]
assert len(problem_dirs) == 1, problem_dirs
problem_dir = problem_dirs[0]
library_pathname = problem_dir / 'libthree-mass-four-spring.so'
assert library_pathname.is_file(), library_pathname
library_pathname

PosixPath('/tmp/compose-physics-validation/c8fb38ba6ceaff018094b31c904c1ba0/libthree-mass-four-spring.so')

## Bind the C ABI through ctypes

In [4]:
library = ctypes.CDLL(str(library_pathname))

library.get_n.restype = ctypes.c_int
library.get_k.restype = ctypes.c_int
library.get_n.argtypes = []
library.get_k.argtypes = []

DOUBLE_POINTER = ctypes.POINTER(ctypes.c_double)
library.compute_residual.restype = None
library.compute_residual.argtypes = [DOUBLE_POINTER, DOUBLE_POINTER]
library.compute_update.restype = None
library.compute_update.argtypes = [DOUBLE_POINTER, DOUBLE_POINTER]

SLOT_COUNT = library.get_n()
RESIDUAL_COUNT = library.get_k()
assert SLOT_COUNT == 13, SLOT_COUNT
assert RESIDUAL_COUNT == 3, RESIDUAL_COUNT

SLOT_NAMES = ['x1','x2','x3','v1','v2','v3','a1','a2','a3','m','k','dt','time']
SLOT_INDEX = {name: index for index, name in enumerate(SLOT_NAMES)}
SLOT_INDEX

{'x1': 0,
 'x2': 1,
 'x3': 2,
 'v1': 3,
 'v2': 4,
 'v3': 5,
 'a1': 6,
 'a2': 7,
 'a3': 8,
 'm': 9,
 'k': 10,
 'dt': 11,
 'time': 12}

In [5]:
def buffer_of(state_dict):
    buffer = np.zeros(SLOT_COUNT, dtype=np.float64)
    for name, value in state_dict.items():
        buffer[SLOT_INDEX[name]] = value
    return buffer

def compute_residual(state):
    out = np.zeros(RESIDUAL_COUNT, dtype=np.float64)
    library.compute_residual(
        state.ctypes.data_as(DOUBLE_POINTER),
        out.ctypes.data_as(DOUBLE_POINTER))
    return out

def compute_update(state):
    out = np.zeros(SLOT_COUNT, dtype=np.float64)
    library.compute_update(
        state.ctypes.data_as(DOUBLE_POINTER),
        out.ctypes.data_as(DOUBLE_POINTER))
    return out

## Eigenmode analysis

Stiffness matrix `K = k * [[2,-1,0],[-1,2,-1],[0,-1,2]]`, mass matrix `M = m * I`. Continuous eigenfrequencies are

$$\omega_j^2 = \frac{2k}{m} \left(1 - \cos\frac{j\pi}{4}\right), \quad j=1,2,3.$$

In [6]:
MASS = 1.0
STIFFNESS = 4.0
DT = 1.0e-3

K_matrix = STIFFNESS * np.array([[2,-1,0],[-1,2,-1],[0,-1,2]], float)
M_matrix = MASS * np.eye(3)

analytic_omega2 = np.array([
    (2 * STIFFNESS / MASS) * (1 - math.cos(j * math.pi / 4))
    for j in (1, 2, 3)])
analytic_omega = np.sqrt(analytic_omega2)

eigvals, eigvecs = np.linalg.eigh(K_matrix / MASS)
numerical_omega = np.sqrt(eigvals)

print('analytic  omega:', analytic_omega)
print('numerical omega:', numerical_omega)
assert np.allclose(np.sort(analytic_omega), np.sort(numerical_omega))

MODE_SHAPES = eigvecs

analytic  omega: [1.53073373 2.82842712 3.69551813]
numerical omega: [1.53073373 2.82842712 3.69551813]


## Step each eigenmode and recover the frequency via FFT

In [7]:
def initial_state_for_mode(mode_index, amplitude=1e-3):
    shape = MODE_SHAPES[:, mode_index]
    positions = amplitude * shape
    accelerations = -(K_matrix @ positions) / MASS
    return buffer_of({
        'x1': positions[0], 'x2': positions[1], 'x3': positions[2],
        'v1': 0.0, 'v2': 0.0, 'v3': 0.0,
        'a1': accelerations[0], 'a2': accelerations[1], 'a3': accelerations[2],
        'm': MASS, 'k': STIFFNESS, 'dt': DT, 'time': 0.0,
    })

STEP_COUNT = 65536

def integrate(state):
    history = np.empty((STEP_COUNT, SLOT_COUNT), dtype=np.float64)
    current = state.copy()
    for step in range(STEP_COUNT):
        history[step] = current
        current = compute_update(current)
    return history

def dominant_frequency(signal, dt):
    detrended = signal - signal.mean()
    spectrum = np.abs(np.fft.rfft(detrended))
    freqs = np.fft.rfftfreq(len(detrended), d=dt)
    peak_index = int(np.argmax(spectrum[1:])) + 1
    if 0 < peak_index < len(spectrum) - 1:
        left, center, right = spectrum[peak_index - 1: peak_index + 2]
        denominator = (left - 2 * center + right)
        offset = 0.5 * (left - right) / denominator if denominator != 0 else 0.0
    else:
        offset = 0.0
    bin_width = freqs[1] - freqs[0]
    return freqs[peak_index] + offset * bin_width

histories = []
recovered_omega = np.zeros(3)
for mode_index in range(3):
    history = integrate(initial_state_for_mode(mode_index))
    histories.append(history)
    f_hz = dominant_frequency(history[:, SLOT_INDEX['x1']], DT)
    recovered_omega[mode_index] = 2 * math.pi * f_hz

for mode_index in range(3):
    expected = analytic_omega[mode_index]
    found    = recovered_omega[mode_index]
    print(f'mode {mode_index}: analytic={expected:.6f} '
          f'recovered={found:.6f} '
          f'rel_err={abs(found-expected)/expected:.2e}')
    assert abs(found - expected) / expected < 5e-3

mode 0: analytic=1.530734 recovered=1.533872 rel_err=2.05e-03
mode 1: analytic=2.828427 recovered=2.831494 rel_err=1.08e-03
mode 2: analytic=3.695518 recovered=3.710918 rel_err=4.17e-03


## Residual at machine zero on every step

In [8]:
MAX_RESIDUAL = 0.0
for history in histories:
    for step in range(0, STEP_COUNT, 64):
        r = compute_residual(history[step])
        MAX_RESIDUAL = max(MAX_RESIDUAL, float(np.abs(r).max()))
print('max |residual| over all sampled states:', MAX_RESIDUAL)
assert MAX_RESIDUAL < 1e-10

max |residual| over all sampled states: 1.734723475976807e-18


## Lisp-vs-C bit identity on a sample of states

In [9]:
import tempfile, textwrap

def lisp_evaluate_problem(states):
    """Run SBCL to funcall residual and update rows on each state.

    States are passed as full-precision text (~,17,,,,,'eE) so the
    Lisp reader recovers each double exactly. Outputs come back the
    same way, then we compare uint64 views of the bit patterns.
    """
    nstates = len(states)
    with tempfile.TemporaryDirectory(prefix='compose-physics-eval-') as tmp:
        tmp_path = Path(tmp)
        input_path  = tmp_path / 'input.txt'
        output_path = tmp_path / 'output.txt'
        with input_path.open('w') as handle:
            for state in states:
                handle.write(' '.join(repr(float(value)) for value in state))
                handle.write('\n')
        script = tmp_path / 'eval.lisp'
        script.write_text(textwrap.dedent(f'''\
          (require :asdf)
          (push #p"{PROJECT_ROOT}/" asdf:*central-registry*)
          (asdf:load-system :compose-physics)
          (load #p"{PROBLEM_LISP}")
          (setf *read-default-float-format* (quote double-float))
          (let* ((problem (symbol-value (find-symbol "*PROBLEM*" :common-lisp-user)))
                 (slot-count   (compose-physics:problem-slot-count problem))
                 (residual-rows (compose-physics:problem-residual-rows problem))
                 (update-rows   (compose-physics:problem-update-rows  problem))
                 (residual-count (length residual-rows)))
            (with-open-file (in #p"{input_path}" :direction :input)
              (with-open-file (out #p"{output_path}" :direction :output
                                                     :if-exists :supersede
                                                     :if-does-not-exist :create)
                (loop
                  for line = (read-line in nil nil)
                  while line do
                  (let ((buffer (make-array slot-count :element-type (quote double-float)))
                        (token-stream (make-string-input-stream line)))
                    (loop for i below slot-count do
                          (setf (aref buffer i) (coerce (read token-stream) (quote double-float))))
                    (let ((residual-out (make-array residual-count :element-type (quote double-float)))
                          (update-out (make-array slot-count :element-type (quote double-float))))
                      (loop for row across residual-rows for index from 0 do
                            (setf (aref residual-out index)
                                  (funcall (compose-physics:residual-row-expression row) buffer)))
                      (loop for row across update-rows for index from 0 do
                            (setf (aref update-out (compose-physics:update-row-slot-index row))
                                  (funcall (compose-physics:update-row-expression row) buffer)))
                      (loop for v across residual-out do
                            (format out "~,17,,,,,'eE " v))
                      (loop for v across update-out do
                            (format out "~,17,,,,,'eE " v))
                      (terpri out))))))) '''))
        completed = subprocess.run(['sbcl','--non-interactive','--load',str(script)],
                                   capture_output=True, text=True)
        if completed.returncode != 0:
            raise RuntimeError(completed.stderr)
        residual_out = np.zeros((nstates, RESIDUAL_COUNT), dtype=np.float64)
        update_out   = np.zeros((nstates, SLOT_COUNT),     dtype=np.float64)
        with output_path.open() as handle:
            for row_index, line in enumerate(handle):
                values = [float(token) for token in line.split()]
                residual_out[row_index] = values[:RESIDUAL_COUNT]
                update_out[row_index]   = values[RESIDUAL_COUNT:RESIDUAL_COUNT + SLOT_COUNT]
        return residual_out, update_out

rng = np.random.default_rng(0xC0FFEE)
sample_states = []
for _ in range(8):
    sample_states.append(buffer_of({
        'x1': rng.normal(), 'x2': rng.normal(), 'x3': rng.normal(),
        'v1': rng.normal(), 'v2': rng.normal(), 'v3': rng.normal(),
        'a1': rng.normal(), 'a2': rng.normal(), 'a3': rng.normal(),
        'm': 1.0 + abs(rng.normal()),
        'k': 1.0 + abs(rng.normal()),
        'dt': 1e-3, 'time': rng.normal(),
    }))
sample_array = np.stack(sample_states)
lisp_residual, lisp_update = lisp_evaluate_problem(sample_array)
c_residual = np.stack([compute_residual(s) for s in sample_array])
c_update   = np.stack([compute_update(s)   for s in sample_array])

residual_match = np.array_equal(lisp_residual.view(np.uint64), c_residual.view(np.uint64))
update_match   = np.array_equal(lisp_update.view(np.uint64),   c_update.view(np.uint64))
print('residual bit-identity:', residual_match)
print('update   bit-identity:', update_match)
if not residual_match:
    print('residual max |diff|:', np.abs(lisp_residual - c_residual).max())
if not update_match:
    print('update   max |diff|:', np.abs(lisp_update   - c_update).max())
assert residual_match and update_match

residual bit-identity: True
update   bit-identity: True


## Energy bound under symplectic Euler

In [10]:
def total_energy(state):
    positions = state[[SLOT_INDEX[k] for k in ('x1','x2','x3')]]
    velocities = state[[SLOT_INDEX[k] for k in ('v1','v2','v3')]]
    kinetic = 0.5 * MASS * float(velocities @ velocities)
    potential = 0.5 * float(positions @ K_matrix @ positions)
    return kinetic + potential

rng = np.random.default_rng(0)
general_initial = buffer_of({
    'x1': 1.0e-3, 'x2': -2.0e-3, 'x3': 1.5e-3,
    'v1': 0.0, 'v2': 0.0, 'v3': 0.0,
    'a1': 0.0, 'a2': 0.0, 'a3': 0.0,
    'm': MASS, 'k': STIFFNESS, 'dt': DT, 'time': 0.0,
})
positions0 = general_initial[[SLOT_INDEX[k] for k in ('x1','x2','x3')]]
accelerations0 = -(K_matrix @ positions0) / MASS
for index, name in zip(accelerations0, ('a1','a2','a3')):
    general_initial[SLOT_INDEX[name]] = index

general_history = integrate(general_initial)
energies = np.array([total_energy(general_history[step]) for step in range(0, STEP_COUNT, 16)])
energy_drift = (energies.max() - energies.min()) / energies.mean()
print('relative energy oscillation:', energy_drift)
assert energy_drift < 5e-3

relative energy oscillation: 0.003684772702219734


## Trajectory plots

In [11]:
times = np.arange(STEP_COUNT) * DT
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for axis, slot_group, title in zip(
        axes,
        (('x1','x2','x3'), ('v1','v2','v3'), ('a1','a2','a3')),
        ('positions', 'velocities', 'accelerations')):
    for name in slot_group:
        axis.plot(times, general_history[:, SLOT_INDEX[name]], label=name)
    axis.set_ylabel(title); axis.legend(loc='upper right'); axis.grid(True)
axes[-1].set_xlabel('time')
plt.tight_layout(); plt.show()

kinetic_history = 0.5 * MASS * (
    general_history[:, SLOT_INDEX['v1']]**2
    + general_history[:, SLOT_INDEX['v2']]**2
    + general_history[:, SLOT_INDEX['v3']]**2)
potential_history = np.array([
    0.5 * general_history[step, [SLOT_INDEX[k] for k in ('x1','x2','x3')]]
        @ K_matrix
        @ general_history[step, [SLOT_INDEX[k] for k in ('x1','x2','x3')]]
    for step in range(STEP_COUNT)])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(times, kinetic_history, label='kinetic')
ax.plot(times, potential_history, label='potential')
ax.plot(times, kinetic_history + potential_history, label='total')
ax.set_xlabel('time'); ax.set_ylabel('energy'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_117690/2661416088.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


/tmp/ipykernel_117690/2661416088.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


All four invariants pass. The rewrite is end-to-end correct.